# 02 — Pełna kohorta TCGA-LUAD: sanity check pipeline'u

Notebook do **testowania działania pipeline na pełnej kohorcie**. Pokazuje krok po kroku:

1. Co jest w `data/raw/` (inwentaryzacja wejścia)
2. Walidacja kohorty (QC report — co jest spójne, co nie)
3. Build expression matrix (deduplikacja, ile próbek po stronie ekspresji)
4. Build survival dataset (filtrowanie pacjentów bez survival, próbek bez clinical)
5. **Funnel** — pełen lejek od 601 plików do finalnego datasetu, krok po kroku ile odpadło i dlaczego
6. Sanity check finalnego datasetu — eventy, czas obserwacji, staging, demografia

W przeciwieństwie do `01_explore_first_samples.ipynb` (który sprawdzał biologię na 2 próbkach), ten notebook **odpowiada na pytanie operacyjne**: czy pipeline da się odpalić end-to-end na realnych 601 plikach z GDC i co z niego wychodzi.


In [3]:
import sys
import json
from pathlib import Path
from datetime import datetime, timezone

import polars as pl

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.ingest import (
    parse_clinical,
    parse_sample_sheet,
    extract_star_file_stem,
    STAR_FILE_PATTERNS,
)
from src.transform import build_expression_matrix, build_survival_dataset
from src.validate import (
    run_cohort_qc,
    discover_stems,
    save_qc_report,
    Severity,
    QCCategory,
)

print(f"Projekt: {PROJECT_ROOT}")
print(f"Polars: {pl.__version__}")
print(f"Czas: {datetime.now(timezone.utc).isoformat()}")


Projekt: /Users/luka/luad-huba-clean
Polars: 1.40.1
Czas: 2026-06-05T11:57:17.949165+00:00


## 1. Konfiguracja ścieżek

Pipeline domyślnie czyta z `data/raw/` i pisze do `data/interim/` oraz `data/processed/`. Notebook nie nadpisuje danych — jeśli pliki interim już są (po wcześniejszym `parse-star`), używa ich.


In [4]:
RAW_DIR = PROJECT_ROOT / "data" / "raw"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim" / "star_counts"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
LOGS_DIR = PROJECT_ROOT / "logs" / "qc"

for d in [RAW_DIR, INTERIM_DIR, PROCESSED_DIR, LOGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"RAW:       {RAW_DIR}")
print(f"INTERIM:   {INTERIM_DIR}")
print(f"PROCESSED: {PROCESSED_DIR}")
print(f"LOGS:      {LOGS_DIR}")


RAW:       /Users/luka/luad-huba-clean/data/raw
INTERIM:   /Users/luka/luad-huba-clean/data/interim/star_counts
PROCESSED: /Users/luka/luad-huba-clean/data/processed
LOGS:      /Users/luka/luad-huba-clean/logs/qc


## 2. Inwentaryzacja `data/raw/`

Ile plików STAR, ile pacjentów w clinical, ile rekordów w sample sheet. To jest punkt zerowy lejka.


In [5]:
star_files = []
for pattern in STAR_FILE_PATTERNS:
    star_files.extend(RAW_DIR.rglob(pattern))
star_files = sorted(set(star_files))

sheet_path = next(RAW_DIR.rglob("gdc_sample_sheet*.tsv"), None)
clinical_path = next(RAW_DIR.rglob("clinical.tsv"), None)
metadata_path = next(RAW_DIR.rglob("metadata*.json"), None)

print(f"Pliki STAR-Counts:   {len(star_files)}")
print(f"Sample sheet:        {sheet_path.name if sheet_path else 'BRAK'}")
print(f"Clinical:            {clinical_path.name if clinical_path else 'BRAK'}")
print(f"Metadata JSON:       {metadata_path.name if metadata_path else 'BRAK'}")

if not (sheet_path and clinical_path and star_files):
    print()
    print("UWAGA: brakuje wymaganych plików. Notebook nie ruszy dalej.")


Pliki STAR-Counts:   601
Sample sheet:        gdc_sample_sheet.2026-05-30.tsv
Clinical:            clinical.tsv
Metadata JSON:       metadata.cart.2026-05-30.json


## 3. Wczytanie sample sheet i clinical

Parsery są źródłem prawdy o tym, ile mamy próbek (sheet) i pacjentów (clinical) **po odfiltrowaniu** tych, których nie da się użyć.


In [6]:
sheet = parse_sample_sheet(sheet_path)
print(f"Sample sheet po parserze: {sheet.height} próbek, {sheet.width} kolumn")
print()
print("Rozkład tissue_type:")
print(sheet.group_by("tissue_type").len().sort("len", descending=True))
print()
print("Próbki tumor vs normal:")
print(sheet.group_by("is_tumor").len())


Sample sheet po parserze: 601 próbek, 7 kolumn

Rozkład tissue_type:
shape: (2, 2)
┌─────────────┬─────┐
│ tissue_type ┆ len │
│ ---         ┆ --- │
│ str         ┆ u32 │
╞═════════════╪═════╡
│ Tumor       ┆ 542 │
│ Normal      ┆ 59  │
└─────────────┴─────┘

Próbki tumor vs normal:
shape: (2, 2)
┌──────────┬─────┐
│ is_tumor ┆ len │
│ ---      ┆ --- │
│ bool     ┆ u32 │
╞══════════╪═════╡
│ false    ┆ 59  │
│ true     ┆ 542 │
└──────────┴─────┘


In [7]:
print("Parsowanie clinical (filter pacjentów bez survival może wypisać warning na stderr)...")
print()
clinical = parse_clinical(clinical_path)
print(f"Clinical po parserze: {clinical.height} pacjentów, {clinical.width} kolumn")
print()
print("Rozkład vital_status:")
print(clinical.group_by("vital_status").len())
print()
print("Eventy (śmierć):")
n_events = clinical.filter(pl.col("event")).height
print(f"  Dead:  {n_events}")
print(f"  Alive: {clinical.height - n_events}")


Parsowanie clinical (filter pacjentów bez survival może wypisać warning na stderr)...

Clinical po parserze: 509 pacjentów, 11 kolumn

Rozkład vital_status:
shape: (2, 2)
┌──────────────┬─────┐
│ vital_status ┆ len │
│ ---          ┆ --- │
│ str          ┆ u32 │
╞══════════════╪═════╡
│ Alive        ┆ 326 │
│ Dead         ┆ 183 │
└──────────────┴─────┘

Eventy (śmierć):
  Dead:  183
  Alive: 326


clinical_parser: pominięto 9 pacjentów bez czasu obserwacji w clinical.tsv (przykłady: ['TCGA-75-7031', 'TCGA-80-5607', 'TCGA-75-6203']). Sprawdź days_to_death oraz days_to_last_follow_up.


## 4. Walidacja spójności kohorty (QC report)

Cztery reguły:
- **MISSING_STAR_FILE** (ERROR) — próbka w sheecie, brak pliku
- **ORPHAN_STAR_FILE** (WARNING) — plik na dysku, brak w sheecie
- **MISSING_CLINICAL** (ERROR) — pacjent bez danych klinicznych
- **DUPLICATE_SAMPLE** (WARNING) — ten sam `sample_id` wielokrotnie w sheecie

Raport jest też zapisywany jako JSON do `logs/qc/`.


In [8]:
available_stems = {extract_star_file_stem(p.name) for p in star_files}

qc_report = run_cohort_qc(
    sample_sheet=sheet,
    clinical=clinical,
    available_stems=available_stems,
)

print("=== PODSUMOWANIE QC ===")
summary = qc_report.summary()
print(f"Łącznie problemów: {summary['total']}")
print(f"  ERROR:   {summary['errors']}")
print(f"  WARNING: {summary['warnings']}")
print(f"  INFO:    {summary['info']}")
print()

print("=== Rozkład per kategoria ===")
for category in QCCategory:
    issues = qc_report.by_category(category)
    if issues:
        print(f"{category.name:25} {len(issues):>5}")


=== PODSUMOWANIE QC ===
Łącznie problemów: 20
  ERROR:   9
  INFO:    0

=== Rozkład per kategoria ===
MISSING_CLINICAL              9
DUPLICATE_SAMPLE             11


In [9]:
print("=== Przykłady ERROR (max 5) ===")
errors = qc_report.by_severity(Severity.ERROR)
for issue in errors[:5]:
    print(f"  [{issue.category.name}] {issue.message}")
    if issue.context:
        print(f"      context: {issue.context}")

print()
print("=== Przykłady WARNING (max 5) ===")
warnings = qc_report.by_severity(Severity.WARNING)
for issue in warnings[:5]:
    print(f"  [{issue.category.name}] {issue.message}")
    if issue.context:
        print(f"      context: {issue.context}")


=== Przykłady ERROR (max 5) ===
  [MISSING_CLINICAL] Pacjent TCGA-75-5122 (próbka TCGA-75-5122-01A) nie ma danych klinicznych
      context: {'case_id': 'TCGA-75-5122', 'sample_id': 'TCGA-75-5122-01A'}
  [MISSING_CLINICAL] Pacjent TCGA-75-5126 (próbka TCGA-75-5126-01A) nie ma danych klinicznych
      context: {'case_id': 'TCGA-75-5126', 'sample_id': 'TCGA-75-5126-01A'}
  [MISSING_CLINICAL] Pacjent TCGA-80-5607 (próbka TCGA-80-5607-01A) nie ma danych klinicznych
      context: {'case_id': 'TCGA-80-5607', 'sample_id': 'TCGA-80-5607-01A'}
  [MISSING_CLINICAL] Pacjent TCGA-75-7031 (próbka TCGA-75-7031-01A) nie ma danych klinicznych
      context: {'case_id': 'TCGA-75-7031', 'sample_id': 'TCGA-75-7031-01A'}
  [MISSING_CLINICAL] Pacjent TCGA-75-6207 (próbka TCGA-75-6207-01A) nie ma danych klinicznych
      context: {'case_id': 'TCGA-75-6207', 'sample_id': 'TCGA-75-6207-01A'}

=== Przykłady WARNING (max 5) ===
  [DUPLICATE_SAMPLE] Próbka TCGA-44-2656-01A występuje 2 razy
      context: {'samp

In [10]:
report_path = save_qc_report(qc_report, LOGS_DIR)
print(f"Raport zapisany: {report_path}")


Raport zapisany: /Users/luka/luad-huba-clean/logs/qc/qc_report_20260605T115720Z.json


## 5. Build expression matrix

Buduje macierz `geny × próbki` ze wszystkich parquetów z `data/interim/star_counts/`. Domyślna strategia deduplikacji to `deepest` — dla próbek z wieloma plikami wybiera ten z największą sumą odczytów.

**Uwaga:** ten krok wymaga, żeby parquety były już utworzone (`luad-huba parse-star`). Jeśli ich nie ma, notebook to wykryje i pominie.


In [11]:
parquets = sorted(INTERIM_DIR.rglob("*.parquet"))
print(f"Plików parquet w {INTERIM_DIR.relative_to(PROJECT_ROOT)}: {len(parquets)}")

if len(parquets) == 0:
    print()
    print("Brak parquetów - odpal najpierw: luad-huba parse-star")
    matrix = None
elif len(parquets) != len(star_files):
    print(f"UWAGA: liczba parquetów ({len(parquets)}) != liczba TSV ({len(star_files)})")
    print("Niektóre pliki mogły nie zostać sparsowane")
    matrix = None
else:
    matrix = build_expression_matrix(
        parquet_paths=parquets,
        sample_sheet=sheet,
        metric="unstranded",
        duplicate_strategy="deepest",
    )
    print()
    print(f"Macierz: {matrix.height} genów × {matrix.width - 1} próbek")
    print(f"Liczba próbek w sheecie:    {sheet.height}")
    print(f"Liczba próbek w macierzy:   {matrix.width - 1}")
    print(f"Różnica (deduplikacja):     {sheet.height - (matrix.width - 1)}")


Plików parquet w data/interim/star_counts: 601

Macierz: 60660 genów × 590 próbek
Liczba próbek w sheecie:    601
Liczba próbek w macierzy:   590
Różnica (deduplikacja):     11


## 6. Build survival dataset

Tu zachodzi finalna integracja:
- macierz transponowana (próbki → wiersze)
- doklejone kowarianty kliniczne (`time`, `event`, `age`, `gender`, `stage`)
- filtr `tumor_only=True` (zostają tylko próbki nowotworowe)
- filter+log próbek, dla których nie ma dopasowania klinicznego (efekt wcześniejszego filtrowania pacjentów bez survival w clinical_parser)

Komunikaty na stderr pokażą dokładnie, ile próbek odpadło i dlaczego.


In [12]:
if matrix is not None:
    print("Budowanie survival dataset (komunikaty filtra pojawią się na stderr)...")
    print()
    dataset = build_survival_dataset(
        expression_matrix=matrix,
        sample_sheet=sheet,
        clinical=clinical,
        tumor_only=True,
    )
    print()
    print(f"Finalny dataset: {dataset.height} próbek × {dataset.width} kolumn")
    print(f"  ({len(['sample_id', 'case_id'] + ['time', 'event', 'age_at_index', 'gender', 'ajcc_pathologic_stage', 'tissue_type'])} kolumn metadanych "
          f"+ {dataset.width - 8} kolumn genów)")
else:
    dataset = None
    print("Pomijam - brak macierzy ekspresji")


Budowanie survival dataset (komunikaty filtra pojawią się na stderr)...


Finalny dataset: 533 próbek × 60668 kolumn
  (8 kolumn metadanych + 60660 kolumn genów)


survival_dataset: pominięto 9 próbek bez dopasowania w danych klinicznych (9 unikalnych pacjentów, przykłady: ['TCGA-75-7030', 'TCGA-80-5607', 'TCGA-75-6211'])


## 7. Funnel — pełen lejek przetwarzania

Tabela pokazująca **gdzie i ile próbek/pacjentów odpada na każdym etapie**. To jest najważniejszy diagram dla obrony projektu i samego zrozumienia jakości danych.


In [13]:
rows = []

rows.append({
    "Etap": "1. Pliki STAR w data/raw/",
    "Liczba": len(star_files),
    "Komentarz": "Pobrane z GDC",
})

rows.append({
    "Etap": "2. Rekordy w sample sheet",
    "Liczba": sheet.height,
    "Komentarz": "Po parserze (waliduje UUID, dodaje tcga_sample_code)",
})

rows.append({
    "Etap": "3. Pacjenci w clinical (po parserze)",
    "Liczba": clinical.height,
    "Komentarz": "Po dedupie po primary_disease i filtrze no-survival",
})

if matrix is not None:
    n_in_matrix = matrix.width - 1
    rows.append({
        "Etap": "4. Próbki w expression matrix",
        "Liczba": n_in_matrix,
        "Komentarz": f"Po deduplikacji (strategy=deepest): -{sheet.height - n_in_matrix}",
    })

if dataset is not None:
    rows.append({
        "Etap": "5. Próbki tumor w macierzy",
        "Liczba": sheet.filter(pl.col("is_tumor")).height,
        "Komentarz": "Próbki normal odfiltrowane (tumor_only=True)",
    })
    rows.append({
        "Etap": "6. Finalny survival dataset",
        "Liczba": dataset.height,
        "Komentarz": "Po filtrze próbek bez clinical (efekt kroku 3)",
    })
    n_events = dataset.filter(pl.col("event")).height
    rows.append({
        "Etap": "7. Z tego: eventy (śmierć)",
        "Liczba": n_events,
        "Komentarz": f"Censoring rate: {(1 - n_events/dataset.height)*100:.1f}%",
    })

funnel = pl.DataFrame(rows)
print(funnel)


shape: (7, 3)
┌─────────────────────────────────┬────────┬─────────────────────────────────┐
│ Etap                            ┆ Liczba ┆ Komentarz                       │
│ ---                             ┆ ---    ┆ ---                             │
│ str                             ┆ i64    ┆ str                             │
╞═════════════════════════════════╪════════╪═════════════════════════════════╡
│ 1. Pliki STAR w data/raw/       ┆ 601    ┆ Pobrane z GDC                   │
│ 2. Rekordy w sample sheet       ┆ 601    ┆ Po parserze (waliduje UUID, do… │
│ 3. Pacjenci w clinical (po par… ┆ 509    ┆ Po dedupie po primary_disease … │
│ 4. Próbki w expression matrix   ┆ 590    ┆ Po deduplikacji (strategy=deep… │
│ 5. Próbki tumor w macierzy      ┆ 542    ┆ Próbki normal odfiltrowane (tu… │
│ 6. Finalny survival dataset     ┆ 533    ┆ Po filtrze próbek bez clinical… │
│ 7. Z tego: eventy (śmierć)      ┆ 189    ┆ Censoring rate: 64.5%           │
└─────────────────────────────────┴───

## 8. Sanity check finalnego datasetu

Czy finalny dataset ma sens? Rozkłady kluczowych zmiennych.


In [14]:
if dataset is not None:
    print("=== EVENT / CENSORING ===")
    print(dataset.group_by("event").len())
    print()

    print("=== ROZKŁAD CZASU OBSERWACJI (dni) ===")
    print(dataset.select([
        pl.col("time").min().alias("min"),
        pl.col("time").quantile(0.25).alias("q25"),
        pl.col("time").median().alias("median"),
        pl.col("time").quantile(0.75).alias("q75"),
        pl.col("time").max().alias("max"),
    ]))
    print()

    print("=== ROZKŁAD STAGE ===")
    print(dataset.group_by("ajcc_pathologic_stage").len().sort("len", descending=True))
    print()

    print("=== ROZKŁAD GENDER ===")
    print(dataset.group_by("gender").len())
    print()

    print("=== WIEK W MOMENCIE DIAGNOZY ===")
    print(dataset.select([
        pl.col("age_at_index").min().alias("min"),
        pl.col("age_at_index").mean().alias("mean"),
        pl.col("age_at_index").max().alias("max"),
    ]))
else:
    print("Brak datasetu - pomijam sanity check")


=== EVENT / CENSORING ===
shape: (2, 2)
┌───────┬─────┐
│ event ┆ len │
│ ---   ┆ --- │
│ bool  ┆ u32 │
╞═══════╪═════╡
│ false ┆ 344 │
│ true  ┆ 189 │
└───────┴─────┘

=== ROZKŁAD CZASU OBSERWACJI (dni) ===
shape: (1, 5)
┌─────┬───────┬────────┬────────┬──────┐
│ min ┆ q25   ┆ median ┆ q75    ┆ max  │
│ --- ┆ ---   ┆ ---    ┆ ---    ┆ ---  │
│ i64 ┆ f64   ┆ f64    ┆ f64    ┆ i64  │
╞═════╪═══════╪════════╪════════╪══════╡
│ 0   ┆ 424.0 ┆ 677.0  ┆ 1147.0 ┆ 7248 │
└─────┴───────┴────────┴────────┴──────┘

=== ROZKŁAD STAGE ===
shape: (10, 2)
┌───────────────────────┬─────┐
│ ajcc_pathologic_stage ┆ len │
│ ---                   ┆ --- │
│ str                   ┆ u32 │
╞═══════════════════════╪═════╡
│ Stage IB              ┆ 150 │
│ Stage IA              ┆ 138 │
│ Stage IIB             ┆ 73  │
│ Stage IIIA            ┆ 71  │
│ Stage IIA             ┆ 50  │
│ Stage IV              ┆ 26  │
│ Stage IIIB            ┆ 11  │
│ null                  ┆ 8   │
│ Stage I               ┆ 5   │
│ Sta

## 9. Eventy per stage — czy mamy moc statystyczną?

Dla modeli przeżywalności kluczowe jest, ile eventów (śmierci) mamy w każdej grupie. Stage IV z 3 eventami nie pociągnie sensownej analizy stratyfikowanej.


In [15]:
if dataset is not None:
    events_by_stage = (
        dataset
        .group_by("ajcc_pathologic_stage")
        .agg([
            pl.len().alias("n_samples"),
            pl.col("event").sum().alias("n_events"),
        ])
        .with_columns(
            (pl.col("n_events") / pl.col("n_samples") * 100).round(1).alias("event_rate_pct")
        )
        .sort("ajcc_pathologic_stage")
    )
    print(events_by_stage)
else:
    print("Brak datasetu")


shape: (10, 4)
┌───────────────────────┬───────────┬──────────┬────────────────┐
│ ajcc_pathologic_stage ┆ n_samples ┆ n_events ┆ event_rate_pct │
│ ---                   ┆ ---       ┆ ---      ┆ ---            │
│ str                   ┆ u32       ┆ u32      ┆ f64            │
╞═══════════════════════╪═══════════╪══════════╪════════════════╡
│ null                  ┆ 8         ┆ 2        ┆ 25.0           │
│ Stage I               ┆ 5         ┆ 1        ┆ 20.0           │
│ Stage IA              ┆ 138       ┆ 28       ┆ 20.3           │
│ Stage IB              ┆ 150       ┆ 42       ┆ 28.0           │
│ Stage II              ┆ 1         ┆ 1        ┆ 100.0          │
│ Stage IIA             ┆ 50        ┆ 21       ┆ 42.0           │
│ Stage IIB             ┆ 73        ┆ 32       ┆ 43.8           │
│ Stage IIIA            ┆ 71        ┆ 39       ┆ 54.9           │
│ Stage IIIB            ┆ 11        ┆ 7        ┆ 63.6           │
│ Stage IV              ┆ 26        ┆ 16       ┆ 61.5        

## 10. Deep dive — rzeczy, które warto sprawdzić dokładniej

Funnel pokazuje *ile* odpadło. Ta sekcja pokazuje *kto* dokładnie i *dlaczego*.
Pomaga wyłapać podejrzane wzorce w danych przed wejściem w modelowanie.


### 10.1 Rozkład tumor vs normal **po dedupe**

Funnel mówi, że 11 plików odpadło przy deduplikacji (601 → 590). Ale czy to były głównie próbki tumor, czy normal? To zmienia interpretację — jeśli wszystkie duplikaty pochodzą z tkanki prawidłowej, deduplikacja nie wpływa na cohort modelowy.


In [16]:
if matrix is not None:
    matrix_sample_ids = [c for c in matrix.columns if c != "gene_id"]
    post_dedupe_sheet = sheet.filter(pl.col("sample_id").is_in(matrix_sample_ids))

    print("Rozkład tissue_type po deduplikacji (590 unique sample_ids):")
    print(post_dedupe_sheet.group_by("tissue_type").len().sort("len", descending=True))
    print()

    print("Dla porównania - przed deduplikacją (601 rekordów w sheecie):")
    print(sheet.group_by("tissue_type").len().sort("len", descending=True))
    print()

    pre_tumor = sheet.filter(pl.col("is_tumor")).height
    post_tumor = post_dedupe_sheet.filter(pl.col("is_tumor")).height
    pre_normal = sheet.height - pre_tumor
    post_normal = post_dedupe_sheet.height - post_tumor

    print(f"Tumor:  {pre_tumor} -> {post_tumor} (dedupe usunął {pre_tumor - post_tumor})")
    print(f"Normal: {pre_normal} -> {post_normal} (dedupe usunął {pre_normal - post_normal})")


Rozkład tissue_type po deduplikacji (590 unique sample_ids):
shape: (2, 2)
┌─────────────┬─────┐
│ tissue_type ┆ len │
│ ---         ┆ --- │
│ str         ┆ u32 │
╞═════════════╪═════╡
│ Tumor       ┆ 542 │
│ Normal      ┆ 59  │
└─────────────┴─────┘

Dla porównania - przed deduplikacją (601 rekordów w sheecie):
shape: (2, 2)
┌─────────────┬─────┐
│ tissue_type ┆ len │
│ ---         ┆ --- │
│ str         ┆ u32 │
╞═════════════╪═════╡
│ Tumor       ┆ 542 │
│ Normal      ┆ 59  │
└─────────────┴─────┘

Tumor:  542 -> 542 (dedupe usunął 0)
Normal: 59 -> 59 (dedupe usunął 0)


### 10.2 Kto ma `time = 0` lub bardzo krótki follow-up?

Pacjenci z zerowym lub minimalnym czasem obserwacji nie wnoszą informacji do modeli przeżywalności. `configs/filters.yaml` definiuje `min_follow_up_days: 30`, ale ten próg jeszcze nie jest zintegrowany z kodem — tu pokazujemy, kogo dotyczy.


In [17]:
if dataset is not None:
    short_followup = dataset.filter(pl.col("time") < 30).sort("time")
    print(f"Próbki z time < 30 dni: {short_followup.height}")
    print()
    if short_followup.height > 0:
        print(short_followup.select([
            "sample_id", "case_id", "time", "event",
            "ajcc_pathologic_stage", "vital_status"
        ] if "vital_status" in short_followup.columns else [
            "sample_id", "case_id", "time", "event", "ajcc_pathologic_stage"
        ]))


Próbki z time < 30 dni: 14

shape: (14, 5)
┌──────────────────┬──────────────┬──────┬───────┬───────────────────────┐
│ sample_id        ┆ case_id      ┆ time ┆ event ┆ ajcc_pathologic_stage │
│ ---              ┆ ---          ┆ ---  ┆ ---   ┆ ---                   │
│ str              ┆ str          ┆ i64  ┆ bool  ┆ str                   │
╞══════════════════╪══════════════╪══════╪═══════╪═══════════════════════╡
│ TCGA-05-4244-01A ┆ TCGA-05-4244 ┆ 0    ┆ false ┆ Stage IV              │
│ TCGA-05-4395-01A ┆ TCGA-05-4395 ┆ 0    ┆ true  ┆ Stage IIIB            │
│ TCGA-05-4410-01A ┆ TCGA-05-4410 ┆ 0    ┆ false ┆ Stage IB              │
│ TCGA-86-8281-01A ┆ TCGA-86-8281 ┆ 0    ┆ false ┆ Stage IA              │
│ TCGA-NJ-A4YI-01A ┆ TCGA-NJ-A4YI ┆ 4    ┆ true  ┆ Stage IIIA            │
│ …                ┆ …            ┆ …    ┆ …     ┆ …                     │
│ TCGA-97-7938-01A ┆ TCGA-97-7938 ┆ 18   ┆ true  ┆ Stage IA              │
│ TCGA-86-8672-01A ┆ TCGA-86-8672 ┆ 19   ┆ true  ┆ Stage 

### 10.3 Outliery na drugim końcu — bardzo długi follow-up

Maksimum z opisu statystycznego (`time.max()`) może być nadzwyczajne. TCGA-LUAD zaczął zbierać dane od ~2010, więc realne maksimum follow-upu to ~15 lat (5500 dni). Wartości znacznie wyższe mogą być artefaktem (błąd w danych) lub pacjentem dołączonym z pre-TCGA cohort.


In [18]:
if dataset is not None:
    long_followup = dataset.filter(pl.col("time") > 5000).sort("time", descending=True)
    print(f"Próbki z time > 5000 dni (~13.7 lat): {long_followup.height}")
    print()
    if long_followup.height > 0:
        print(long_followup.select([
            "sample_id", "case_id", "time", "event",
            "ajcc_pathologic_stage", "age_at_index"
        ]))


Próbki z time > 5000 dni (~13.7 lat): 3

shape: (3, 6)
┌──────────────────┬──────────────┬──────┬───────┬───────────────────────┬──────────────┐
│ sample_id        ┆ case_id      ┆ time ┆ event ┆ ajcc_pathologic_stage ┆ age_at_index │
│ ---              ┆ ---          ┆ ---  ┆ ---   ┆ ---                   ┆ ---          │
│ str              ┆ str          ┆ i64  ┆ bool  ┆ str                   ┆ i64          │
╞══════════════════╪══════════════╪══════╪═══════╪═══════════════════════╪══════════════╡
│ TCGA-78-7163-01A ┆ TCGA-78-7163 ┆ 7248 ┆ false ┆ Stage IB              ┆ 60           │
│ TCGA-78-8640-01A ┆ TCGA-78-8640 ┆ 7062 ┆ false ┆ Stage IIA             ┆ 59           │
│ TCGA-49-AARQ-01A ┆ TCGA-49-AARQ ┆ 6732 ┆ false ┆ Stage I               ┆ 41           │
└──────────────────┴──────────────┴──────┴───────┴───────────────────────┴──────────────┘


### 10.4 Pacjenci z wieloma próbkami tumor

W dataset mamy więcej próbek (533) niż unikalnych pacjentów (509). Różnica ~24 to pacjenci, którzy mają wielokrotne próbki tumor — np. różne aliquoty (`01A` + `01B`), albo primary + recurrence (`01A` + `02A`).

Konsekwencja dla survival analysis: **dwie próbki z tego samego pacjenta dziedziczą identyczne `time` i `event`**, więc nie są niezależne. Standardowo do baseline'u wybiera się jedną próbkę per pacjent (typowo `01A`), albo używa modeli z `cluster_col='case_id'` (lifelines wspiera).


In [19]:
if dataset is not None:
    samples_per_case = (
        dataset
        .group_by("case_id")
        .agg([
            pl.len().alias("n_samples"),
            pl.col("sample_id").alias("samples"),
        ])
        .filter(pl.col("n_samples") > 1)
        .sort("n_samples", descending=True)
    )

    print(f"Pacjenci z >1 próbką w datasecie: {samples_per_case.height}")
    print(f"Łącznie nadmiarowych próbek: {samples_per_case['n_samples'].sum() - samples_per_case.height}")
    print()

    print("Top 10 pacjentów z największą liczbą próbek:")
    print(samples_per_case.head(10))


Pacjenci z >1 próbką w datasecie: 14
Łącznie nadmiarowych próbek: 25

Top 10 pacjentów z największą liczbą próbek:
shape: (10, 3)
┌──────────────┬───────────┬─────────────────────────────────┐
│ case_id      ┆ n_samples ┆ samples                         │
│ ---          ┆ ---       ┆ ---                             │
│ str          ┆ u32       ┆ list[str]                       │
╞══════════════╪═══════════╪═════════════════════════════════╡
│ TCGA-44-2662 ┆ 3         ┆ ["TCGA-44-2662-01A", "TCGA-44-… │
│ TCGA-44-3918 ┆ 3         ┆ ["TCGA-44-3918-01A", "TCGA-44-… │
│ TCGA-44-6146 ┆ 3         ┆ ["TCGA-44-6146-01A", "TCGA-44-… │
│ TCGA-44-2666 ┆ 3         ┆ ["TCGA-44-2666-01A", "TCGA-44-… │
│ TCGA-44-4112 ┆ 3         ┆ ["TCGA-44-4112-01A", "TCGA-44-… │
│ TCGA-44-2665 ┆ 3         ┆ ["TCGA-44-2665-01A", "TCGA-44-… │
│ TCGA-44-5645 ┆ 3         ┆ ["TCGA-44-5645-01A", "TCGA-44-… │
│ TCGA-44-6775 ┆ 3         ┆ ["TCGA-44-6775-01A", "TCGA-44-… │
│ TCGA-44-6147 ┆ 3         ┆ ["TCGA-44-6147-01A", "

### 10.5 Czy multi-sample patients to różne aliquoty czy primary+recurrent?

Sufix `01A` / `01B` = ten sam guz pierwotny, różne aliquoty.
Sufix `01A` + `02A` = guz pierwotny + nawrót.
To biologicznie różne sytuacje, warto je rozróżnić.


In [20]:
if dataset is not None and samples_per_case.height > 0:
    multi_sample_cases = samples_per_case["case_id"].to_list()
    multi_sample_details = (
        dataset
        .filter(pl.col("case_id").is_in(multi_sample_cases))
        .select(["case_id", "sample_id"])
        .with_columns(
            pl.col("sample_id").str.extract(r"-(\d{2}[A-Z])$").alias("tcga_code")
        )
        .sort(["case_id", "sample_id"])
    )

    code_distribution = (
        multi_sample_details
        .group_by("case_id")
        .agg(pl.col("tcga_code").sort().str.concat(" + ").alias("kombinacja"))
        .group_by("kombinacja")
        .len()
        .sort("len", descending=True)
    )
    print("Kombinacje TCGA codes wśród pacjentów z wieloma próbkami:")
    print(code_distribution)


Kombinacje TCGA codes wśród pacjentów z wieloma próbkami:
shape: (4, 2)
┌─────────────────┬─────┐
│ kombinacja      ┆ len │
│ ---             ┆ --- │
│ str             ┆ u32 │
╞═════════════════╪═════╡
│ 01A + 01A + 01B ┆ 10  │
│ 01A + 02A       ┆ 2   │
│ 01A + 01B       ┆ 1   │
│ 01A + 01A + 01C ┆ 1   │
└─────────────────┴─────┘


/var/folders/69/yp9cyjqn07g6xrjh38nxblg80000gn/T/ipykernel_4204/998672548.py:16: DeprecationWarning: `str.concat` is deprecated; use `str.join` instead. Note also that the default `delimiter` for `str.join` is an empty string, not a hyphen.
  .agg(pl.col("tcga_code").sort().str.concat(" + ").alias("kombinacja"))


## 11. Wnioski

Po przejściu przez ten notebook wiesz:

- ile plików weszło z GDC i ile faktycznie trafiło do finalnego datasetu
- które próbki/pacjenci odpadli i dlaczego (QC report + filter logs)
- czy duplikaty w TCGA zostały sensownie zdeduplikowane
- czy rozkłady czasów obserwacji, eventów i kowariantów wyglądają zdrowo
- czy mamy dość eventów w każdym stage'u, żeby cokolwiek modelować

**Co dalej:**

1. Jeśli QC zwraca `ERROR` — coś jest nie tak z kohortą, nie idź dalej w analizę
2. Jeśli liczba eventów < ~50, zastanów się nad metodą (Kaplan-Meier działa, ale Cox z wieloma kowariantami nie pociągnie)
3. Finalny dataset (`data/processed/survival_dataset.parquet`) jest gotowy do wzięcia przez `lifelines` lub `scikit-survival`

Następny notebook (planowany): `03_baseline_survival.ipynb` — pierwsze modele Kaplan-Meier i Cox na tym datasecie.
